# Order Prediction — Binary Classification

This notebook predicts whether a customer places an **order** (`order = 1`) for a given product interaction. The model output is a probability that downstream business logic (or a chosen threshold) can act on.

## Approach
- Use the processed dataset, which already merges `items.csv` + `train.csv` and contains the features engineered.
- Drop columns that are leakage, identifiers, or post-decision flags.
- Add a few extra features (`price_to_rrp_ratio`, `log_price`, `day_of_week`, `price_per_unit`, `is_undercutting_competitor`).
- Train a **CatBoost** classifier with native categorical handling on raw string columns.
- Evaluate on a **time-based** test set (last ~2 weeks held out) so the metric reflects predicting the future, not random rows.

## Why CatBoost (not Random Forest or XGBoost)
The other models were already explored priviously. CatBoost differs in three ways that fit our data well:
1. **Native categorical handling on raw strings.** It builds ordered target statistics internally rather than requiring label encoding or one-hot. With `manufacturer` (289 values), `group` (233), `pharmForm` (176), `category` (217), this matters.
2. **Native NaN handling for numeric features.** No imputation needed for `competitorPrice` and the price-vs-competitor ratios.
3. **Tends to need less hyperparameter tuning** to reach a strong baseline than XGBoost.

## Why a time-based split (not random stratified)
The EDA flagged a clear regime shift at day 26 (`is_post_shift_day` feature). A random split lets the model leak signal from "future" days into the training set — the test score then overstates how well the model would generalise. Splitting at day 79 keeps days 1–78 for training (all pre-shift + early post-shift) and days 79–92 for testing (the most recent two weeks). The `is_post_shift_day` flag tells the model which regime each row belongs to, so we keep all data for training without polluting the test set.

# Phase 1: Setup & Sanity Checks

In [13]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import time

from catboost import CatBoostClassifier, Pool
from sklearn.metrics import (
    accuracy_score, roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    roc_curve, precision_recall_curve,
)

FULL_DATA_PATH = "../../../data/processed/processed_joined_dataset.csv"
SAMPLE_DATA_PATH = "../../../data/processed/sample.csv"

USE_FULL_DATA = False
RANDOM_STATE = 42
SPLIT_DAY = 79

data_path = FULL_DATA_PATH if USE_FULL_DATA else SAMPLE_DATA_PATH
print("Using:", data_path)

Using: ../../../data/processed/sample.csv


## Load the processed dataset

Load the joined dataset that already contains the features: `competitorPrice` zeros recoded to NaN, `has_competitor` flag, `pharmForm` normalised, rare manufacturers/groups/categories grouped to `"other"`, the `is_post_shift_day` regime indicator, the `campaignIndex` one-hots, three competitor-price comparison features, and the previous-available-day price difference.

For development we work on the 200k stratified sample (`sample.csv`). Once the pipeline is solid we can switch to the full ~2.75M dataset by flipping `USE_FULL_DATA`.

In [14]:
df = pd.read_csv(data_path, sep="|")
print("Shape:", df.shape)
print(f"Day range: {df['day'].min()} to {df['day'].max()}")
df.head()

Shape: (200000, 29)
Day range: 1 to 92


,lineID,day,pid,adFlag,availability,competitorPrice,click,basket,order,price,...,rrp,has_competitor,campaignIndex_A,campaignIndex_B,campaignIndex_C,price_diff_competitor,price_ratio_competitor,price_pct_diff_competitor,is_post_shift_day,price_diff_vs_previous_available_day
0,978899,39,9624,0,1,17.19,1,0,0,19.89,...,21.51,1,0,0,0,2.70,1.1571,15.71,1,0.0
1,1267035,47,3969,1,1,18.13,1,0,0,20.85,...,26.07,1,0,0,0,2.72,1.1500,15.00,1,0.0
2,297914,14,16633,0,1,15.06,0,0,1,16.45,...,23.98,1,0,1,0,1.39,1.0923,9.23,0,0.0
3,2554963,87,20147,0,1,4.36,1,0,0,5.17,...,5.45,1,0,0,0,0.81,1.1858,18.58,1,0.0
4,2739211,92,14326,0,1,NaN,0,0,1,6.22,...,6.55,0,0,0,0,NaN,NaN,NaN,1,0.0


## Sanity checks

Four things to confirm before any modelling:

1. The class balance of `order` (to know what "good" accuracy means).
2. Where the missing values sit (to know what CatBoost has to handle natively and not).
3. The day-26 change (to know `is_post_shift_day` is doing useful work).
4. The cardinality of the categorical columns (to know why we're not one-hot encoding them).

In [16]:
# 1. Class balance
class_counts = df["order"].value_counts()
class_pct = (df["order"].value_counts(normalize=True) * 100).round(2)
display(pd.DataFrame({"count": class_counts, "percent": class_pct}))

,count,percent
order,,
0,148829,74.41
1,51171,25.59


Moderate imbalance (~74/26). Manageable without resampling.

In [17]:
# 2. Missing values
missing = df.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
display(missing.to_frame("n_missing"))

,n_missing
competitorPrice,7195
price_diff_competitor,7195
price_pct_diff_competitor,7195
price_ratio_competitor,7195
category,6290
price_diff_vs_previous_available_day,2653


competitorPrice missingness propagates into the 3 derived price-vs-competitor features.
CatBoost handles NaN natively for numeric features — no imputation needed.

In [18]:
# 3. Day-26 regime check
pre_rate  = df.loc[df["day"] <  26, "order"].mean()
post_rate = df.loc[df["day"] >= 26, "order"].mean()
print(f"Order rate pre-shift  (day < 26): {pre_rate:.3f}")
print(f"Order rate post-shift (day >= 26): {post_rate:.3f}")
print(f"Drop: {(pre_rate - post_rate) * 100:.1f} percentage points.")

Order rate pre-shift  (day < 26): 0.370
Order rate post-shift (day >= 26): 0.229
Drop: 14.1 percentage points.


In [20]:
# 4. Categorical cardinality
CATEGORICAL_COLS = ["manufacturer", "group", "pharmForm", "category", "unit"]
card = pd.Series({c: df[c].nunique() for c in CATEGORICAL_COLS}, name="n_unique")
display(card.to_frame())

,n_unique
manufacturer,289
group,233
pharmForm,176
category,217
unit,8


High cardinality on 4 of 5 columns. We pass these to CatBoost as cat_features
so it builds ordered target statistics internally rather than us one-hot encoding ~900 sparse columns.

## Establish the trivial baseline

Before training the model, we set the bar with a trivial baseline: **always predict `0`** (no order). Any real model has to clearly beat this.

Three numbers to track:

- **Accuracy** — fraction correct. The trivial baseline gets the majority-class rate (~74%), which sounds high but is meaningless because we never identify a single buyer.
- **ROC-AUC** — discrimination between the two classes across all thresholds. A constant predictor has AUC = 0.5 (no information). This is the metric we actually optimise.
- **Average Precision (AP)** — area under the precision-recall curve. For an always-0 predictor this collapses to the positive-class rate (~26%), which is the floor any useful model has to beat.

In [21]:
y_all = df["order"].values
y_pred_baseline = np.zeros_like(y_all)

baseline_acc = accuracy_score(y_all, y_pred_baseline)
baseline_auc = 0.5  # constant predictor has zero discrimination by definition
baseline_ap  = y_all.mean()

print(f"Baseline accuracy: {baseline_acc:.4f}")
print(f"Baseline ROC-AUC:  {baseline_auc:.4f}")
print(f"Baseline AP:       {baseline_ap:.4f}")

Baseline accuracy: 0.7441
Baseline ROC-AUC:  0.5000
Baseline AP:       0.2559


# Phase 2: Feature Engineering

Three steps:

1. Drop columns that are leakage, identifiers, or no-signal.
2. Add a few features that were flagged as useful in the EDA but are not yet in `processed_joined_dataset.csv`.
3. Set the categorical columns to a form CatBoost can use natively, then build `X_train`, `X_test`, `y_train`, `y_test` with the time-based split.

## Step 1 — Drop columns we shouldn't feed to the model

| Column | Reason to drop |
|---|---|
| `lineID` | Row identifier, no predictive signal. |
| `pid` | Product identifier. Would let the model memorise specific products and not generalise to unseen ones. We force it to rely on actual product attributes (`manufacturer`, `group`, `category`, `pharmForm`, ...). |
| `revenue` | `revenue = price × quantity_sold` — directly derived from the target. Pure leakage. |
| `click`, `basket` | Funnel steps that occur in the same session as the order decision. In practice the model would not have these at the moment we're trying to predict whether the customer will buy. |

In [22]:
DROP_COLS = [
    "lineID", "pid",
    "revenue", "click", "basket",
]

## Step 2 — Engineer five new features

**`price_to_rrp_ratio = price / rrp`**
Price relative to the recommended retail price. The EDA called this out as a top-5 feature in every model previously tested. Captures perceived discount depth — a product at 70% of RRP signals "deal" regardless of absolute price level.

**`log_price = log(price)`**
The price distribution is right-skewed with a long tail up to ~CHF 380. Even tree-based models can benefit from a log transform when the range is this wide, and the price-elasticity analysis in EDA showed a log relationship between price and conversion.

**`day_of_week = day % 7`**
Weekly seasonality. Order rates often differ between weekdays and weekends; without this feature CatBoost can only get to that signal via the raw `day` column, which also encodes long-term trend.

**`price_per_unit = price / content`**
Pack size matters. A high absolute price can still be a unit-cost bargain on a large pack. `content` has a few NaNs after the parsing (PAK and L 125 cases) — CatBoost handles NaN natively, so we don't impute.

**`is_undercutting_competitor = (price < competitorPrice)`**
A clean binary on top of the continuous `price_ratio_competitor`. When `competitorPrice` is NaN the comparison yields `False` → `0`; combined with the existing `has_competitor` flag the model can still distinguish "not undercutting" from "no competitor data".

In [24]:
df["price_to_rrp_ratio"] = df["price"] / df["rrp"]
df["log_price"] = np.log(df["price"].clip(lower=0.01))
df["day_of_week"] = df["day"] % 7
df["price_per_unit"] = df["price"] / df["content"]
df["is_undercutting_competitor"] = (df["price"] < df["competitorPrice"]).astype("int8")

new_feats = ["price_to_rrp_ratio", "log_price", "day_of_week",
             "price_per_unit", "is_undercutting_competitor"]
display(df[new_feats].describe().round(3))

C:\Users\saidf\PycharmProjects\analyticalProjectProductPricing\.venv\Lib\site-packages\pandas\core\nanops.py:1020: RuntimeWarning: invalid value encountered in subtract
  sqr = _ensure_numeric((avg - values) ** 2)


,price_to_rrp_ratio,log_price,day_of_week,price_per_unit,is_undercutting_competitor
count,200000.000,200000.000,200000.000,200000.000,200000.000
mean,0.769,2.285,3.169,inf,0.265
std,0.152,0.835,2.017,NaN,0.441
min,0.055,-2.659,0.000,0.000,0.000
25%,0.686,1.792,1.000,0.109,0.000
50%,0.762,2.287,3.000,0.260,0.000
75%,0.915,2.800,5.000,0.512,1.000
max,1.693,5.926,6.000,inf,1.000


## Step 3 — Categorical handling

We have five categorical columns:

| Column | Unique values |
|---|---|
| `manufacturer` | 289 |
| `group` | 233 |
| `pharmForm` | 176 |
| `category` | 217 |
| `unit` | 8 |

**Why we don't one-hot encode them.** Even after the team's "rare → other" grouping (95% coverage threshold), we still have hundreds of values per column. One-hot encoding all of them would create ~900 new sparse binary columns — slow to train and weaker for tree-based models than a smarter splitting strategy on the raw values.

**What we do instead.** CatBoost accepts a `cat_features` argument and handles the encoding internally using ordered target statistics. We pass the columns as plain strings. Two small fixes first:
- Categorical columns can't contain NaN when passed as `cat_features` — we fill `category`'s NaNs with the string `"unknown"`. The other four columns have no NaN.
- We `.astype(str)` everything to be safe.

In [25]:
for col in CATEGORICAL_COLS:
    df[col] = df[col].fillna("unknown").astype(str)

print("Categorical columns ready:")
display(df[CATEGORICAL_COLS].dtypes.to_frame("dtype"))

Categorical columns ready:


,dtype
manufacturer,str
group,str
pharmForm,str
category,str
unit,str
